In [8]:
# body of the foor loop in the algorithm 3
def NUTS_one_step(L: Callable,
                  theta_0: jnp.ndarray,
                  epsilon: float,
                  key: jnp.ndarray,
                  j_max: int = 10):

    # 1. Sample momentum
    key, sub_r = jrnd.split(key)
    r_0 = jrnd.normal(sub_r, shape=theta_0.shape)

    # 2. Slice variable
    log_joint_0 = log_p(L, theta_0, r_0)
    key, sub_u = jrnd.split(key)
    u = jnp.exp(log_joint_0) * jrnd.uniform(sub_u)

    # 3. Initialize tree
    theta_minus = theta_0
    theta_plus  = theta_0
    r_minus     = r_0
    r_plus      = r_0
    theta_prime = theta_0
    n_prime     = jnp.array(1, dtype=jnp.int32)
    s_prime     = jnp.array(1, dtype=jnp.int32)
    alpha_sum   = jnp.array(0.0)
    n_alpha     = jnp.array(0, dtype=jnp.int32)

    # depth of the tree
    j = 0 

    # 4. Doubling
    while (s_prime == 1) & (j < j_max):

        # Choose a direction
        key, sub_v = jrnd.split(key)
        v_j = jnp.where(jrnd.uniform(sub_v) < 0.5, -1, 1)

        if v_j == -1:
            root = Root(theta_minus, r_minus, u, v_j, j, epsilon, theta_0, r_0)
        else:
            root = Root(theta_plus,  r_plus,  u, v_j, j, epsilon, theta_0, r_0)

        # Call the BuildTree function
        tree, key = BuildTree(L, root, key)

        if v_j == -1:
            theta_minus, r_minus = tree.theta_minus, tree.r_minus
        else:
            theta_plus,  r_plus  = tree.theta_plus,  tree.r_plus

        key, sub = jrnd.split(key)
        total_n = n_prime + tree.n_prime
        p = jnp.where(total_n > 0, tree.n_prime / total_n, 0.5)
        choose = jrnd.bernoulli(sub, p)
        theta_prime = jnp.where(choose, tree.theta_prime, theta_prime)

        n_prime   = n_prime + tree.n_prime
        s_prime   = s_prime * tree.s_prime * stop_criterion(
            theta_minus, theta_plus, r_minus, r_plus
        )
        alpha_sum = alpha_sum + tree.alpha_prime
        n_alpha   = n_alpha + tree.n_a_prime

        j += 1

    accept_rate = alpha_sum / jnp.maximum(1, n_alpha)
    return theta_prime, accept_rate, key


In [9]:
def NUTS_sampler(L: Callable,
                 theta_init: jnp.ndarray,
                 n_samples: int,
                 key: jnp.ndarray,
                 j_max: int = 10):

    epsilon = float(FindReasonableEpsilon(L, theta_init, key))

    thetas = []
    accs = []

    theta = theta_init

    for i in range(n_samples):
        theta, acc, key = NUTS_one_step(L, theta, epsilon, key, j_max=j_max)
        thetas.append(np.array(theta))
        accs.append(float(acc))

    return np.stack(thetas), np.array(accs), epsilon


In [10]:
# Define the pseudo log-potential
# Gaussian log-density
@jax.jit
def L(theta: jnp.ndarray) -> jnp.float32:
    assert theta.ndim == 1
    return -0.5 * jnp.vdot(theta, theta) 

In [11]:
dim = 2
theta0 = jnp.zeros(dim)
key = jrnd.PRNGKey(0)

samples, accs, eps = NUTS_sampler(L, theta0, n_samples=1000, key=key)

print("epsilon:", eps)
print("mean:", samples.mean(axis=0))
print("cov:\n", np.cov(samples.T))
print("mean accept rate:", accs.mean())

epsilon: 0.5
mean: [-0.03251398  0.07115125]
cov:
 [[ 1.18034555 -0.05323093]
 [-0.05323093  1.23084607]]
mean accept rate: 0.9768076473474503
